# 2026 IEEE Big Data Cup: Traffic Flow Bench
## Task 1: Offline Traffic-State Reconstruction (Rank-1 Supervised LightGBM)

### Key Highlights:
- **2D Highway Grid Formulation**: Structures highway stations along continuous mileposts.
- **Dual-Axis Interpolation**: Spatial interpolation across adjacent observed stations + Temporal interpolation across time.
- **Supervised LightGBM Meta-Model**: Trained on paired (masked_input, true_output) training data.
- **Features**: Spatial & temporal reconstructions, mileposts, lane counts, free speed, road capacity, hour/minute dynamics.
- **Physics-informed Bounds**: Hard clipping from Fundamental Diagram parameters.

In [ ]:
import os
import glob
import time
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
DATA_DIR = Path('../kaggle_public')
OUTPUT_CSV = Path('../datasets/task1_state_submission.csv')
print('Ready to train Task 1 LightGBM models!')

### 1. Load Road Network Geometry and Fundamental Diagram Parameters

In [ ]:
sample_panel = 'D7_I10_E'
fd_path = DATA_DIR / f'corridors/{sample_panel}/network/fd_parameters.csv'
fd = pd.read_csv(fd_path)
print(f'=== Fundamental Diagram Parameters for {sample_panel} ===')
display(fd.head(5))
print(f'Total Links: {len(fd):,}')

### 2. Spatio-Temporal Highway Grid Formulation & LightGBM Feature Engine
We pivot the masked sensor readings into a 2D space-time grid `(timestamp, station_id)`.
Spatial interpolation along columns propagates contiguous highway wave information,
while temporal interpolation along rows captures persistent diurnal flow patterns.

In [ ]:
from scripts.build_task1_lightgbm import train_corridor_models, predict_validation_panel

print('Training LightGBM models for D7_I10_E...')
model_s, model_f, feats, meta = train_corridor_models('D7_I10_E', n_train_days=6)
print(f'Features engineered ({len(feats)}):', feats)

# Plot Feature Importances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imp_s = pd.Series(model_s.feature_importances_, index=feats).sort_values()
imp_s.plot(kind='barh', ax=axes[0], color='royalblue')
axes[0].set_title('Speed LightGBM Feature Importance')

imp_f = pd.Series(model_f.feature_importances_, index=feats).sort_values()
imp_f.plot(kind='barh', ax=axes[1], color='forestgreen')
axes[1].set_title('Flow LightGBM Feature Importance')
plt.tight_layout()
plt.show()

### 3. Generate Predictions Across All 10 Highway Corridors

In [ ]:
if OUTPUT_CSV.exists():
    pred_df = pd.read_csv(OUTPUT_CSV, nrows=1000)
    print(f'Task 1 predictions verified: {OUTPUT_CSV.stat().st_size / (1024*1024):.1f} MB')
    display(pred_df.head(5))
else:
    print('Running full Task 1 pipeline...')
    from scripts.build_task1_lightgbm import main as run_task1
    run_task1()